# Классификация текста с помощью нейронных сетей

Используем плотное векторное представление слов (embedding) и полносвязную нейронную сеть.

Чтобы запускать и редактировать код, сохраните копию этого ноутбука себе (Файл -> Создать копию на Диске). Свою копию вы сможете изменять и запускать.

Учебный курс "[Программирование глубоких нейронных сетей на Python](https://openedu.ru/course/urfu/PYDNN/)".

<a target="_blank" href="https://colab.research.google.com/github/sozykin/dlpython_course/blob/master/text_processing/text_classification_pytorch.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



In [1]:
pip install pymorphy3[fast]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import pymorphy3
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from pathlib import Path
import time

In [2]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sozyk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\sozyk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sozyk\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

Константы

In [3]:
max_words = 10000
maxlen = 200
random_state = 42
num_class = 2             # Количество классов
emsize = 64               # Размерность плотного векторного представления
batch_size = 128

## Загружаем и готовим набор данных

In [4]:
data_path = 'data'

In [5]:
path = Path(data_path)

if not path.exists():
    path.mkdir(parents=True, exist_ok=True)

In [ ]:
!curl -s -L -o data/banks.csv "https://www.dropbox.com/scl/fi/mhiwx4plaua183bnuqt5e/train2.csv?rlkey=wv4dalbsutmzu5br9qv5ez21v&dl=1"

In [6]:
banks = pd.read_csv('data/banks.csv', index_col='id');

In [7]:
banks

,text,label
id,,
0,В Альфа-Банке работает замечательная девушка -...,1
1,Оформляя рассрочку в м. Видео в меге тёплый ст...,0
2,Очень порадовала оперативность работы в банке....,1
3,Имела неосторожность оформить потреб. кредит в...,0
4,Небольшая предыстория: Нашел на сайте MDM банк...,0
...,...,...
13994,"О высокой надёжности МКБ, порядочности и добро...",1
13995,"Обслуживаюсь в офисе на Чернореченской 42а, ка...",1
13996,Попала сегодня в очень неприятную ситуацию. Ре...,1


In [8]:
def preprocess(text, stop_words, punctuation_marks, morph):
    tokens = word_tokenize(text.lower())
    preprocessed_text = []
    for token in tokens:
        if token not in punctuation_marks:
            lemma = morph.parse(token)[0].normal_form
            if lemma not in stop_words:
                preprocessed_text.append(lemma)
    return preprocessed_text

In [9]:
punctuation_marks = ['!', ',', '(', ')', ':', '-', '?', '.', '..', '...', '«', '»', ';', '–', '--']
stop_words = stopwords.words("russian")
morph = pymorphy3.MorphAnalyzer()

In [10]:
banks['Preprocessed_texts'] = banks.apply(lambda row: preprocess(row['text'], stop_words, punctuation_marks, morph), axis=1)

In [11]:
banks

,text,label,Preprocessed_texts
id,,,
0,В Альфа-Банке работает замечательная девушка -...,1,"[альфа-банк, работать, замечательный, девушка,..."
1,Оформляя рассрочку в м. Видео в меге тёплый ст...,0,"[оформлять, рассрочка, м., видео, мег, тёплый,..."
2,Очень порадовала оперативность работы в банке....,1,"[очень, порадовать, оперативность, работа, бан..."
3,Имела неосторожность оформить потреб. кредит в...,0,"[иметь, неосторожность, оформить, потреба, кре..."
4,Небольшая предыстория: Нашел на сайте MDM банк...,0,"[небольшой, предыстория, найти, сайт, mdm, бан..."
...,...,...,...
13994,"О высокой надёжности МКБ, порядочности и добро...",1,"[высокий, надёжность, мкб, порядочность, добро..."
13995,"Обслуживаюсь в офисе на Чернореченской 42а, ка...",1,"[обслуживаться, офис, чернореченский, 42а, физ..."
13996,Попала сегодня в очень неприятную ситуацию. Ре...,1,"[попасть, сегодня, очень, неприятный, ситуация..."


Считаем частоту слов во всех отзывах

In [12]:
words = Counter()

In [13]:
for txt in banks['Preprocessed_texts']:
    words.update(txt)

In [14]:
words.most_common(20)

[('банк', 58189),
 ('карта', 30560),
 ('это', 28660),
 ('всё', 20599),
 ('день', 15729),
 ('сотрудник', 14527),
 ('который', 14519),
 ('кредит', 14490),
 ('деньга', 13701),
 ('счёт', 13681),
 ('отделение', 13414),
 ('клиент', 12681),
 ('мочь', 11140),
 ('свой', 11128),
 ('год', 10378),
 ('сказать', 9990),
 ('вопрос', 9638),
 ('ещё', 9606),
 ('очень', 9226),
 ('весь', 9004)]

Создаем словарь, упорядоченный по частоте

В словаре будем использовать 2 специальных кода:
- Код заполнитель: 0
- Неизвестное слово: 1

Нумерация слов в словаре начинается с 2.

In [15]:
# Словарь, отображающий слова в коды
word_to_index = dict()
# Словарь, отображающий коды в слова
index_to_word = dict()

Создаем словари

In [16]:
for i, word in enumerate(words.most_common(max_words - 2)):
    word_to_index[word[0]] = i + 2
    index_to_word[i + 2] = word[0]

In [17]:
word_to_index

{'банк': 2,
 'карта': 3,
 'это': 4,
 'всё': 5,
 'день': 6,
 'сотрудник': 7,
 'который': 8,
 'кредит': 9,
 'деньга': 10,
 'счёт': 11,
 'отделение': 12,
 'клиент': 13,
 'мочь': 14,
 'свой': 15,
 'год': 16,
 'сказать': 17,
 'вопрос': 18,
 'ещё': 19,
 'очень': 20,
 'весь': 21,
 'время': 22,
 'сумма': 23,
 'кредитный': 24,
 'получить': 25,
 'офис': 26,
 'проблема': 27,
 'заявление': 28,
 'договор': 29,
 'работа': 30,
 'платёж': 31,
 'банкомат': 32,
 'телефон': 33,
 'позвонить': 34,
 'месяц': 35,
 'документ': 36,
 'дать': 37,
 'ответ': 38,
 'решить': 39,
 'хотеть': 40,
 'обслуживание': 41,
 'звонить': 42,
 'ваш': 43,
 'работать': 44,
 'услуга': 45,
 'претензия': 46,
 'прийти': 47,
 'вклад': 48,
 'звонок': 49,
 'номер': 50,
 'написать': 51,
 'большой': 52,
 'ситуация': 53,
 'рубль': 54,
 'человек': 55,
 'минута': 56,
 'сделать': 57,
 'просто': 58,
 'говорить': 59,
 'средство': 60,
 'альфа-банк': 61,
 'заявка': 62,
 'срок': 63,
 'очередь': 64,
 '2': 65,
 'первый': 66,
 'знать': 67,
 'информаци

Функция для преобразования списка слов в список кодов

In [18]:
def text_to_sequence(txt, word_to_index):
    seq = []
    for word in txt:
        index = word_to_index.get(word, 1) # 1 означает неизвестное слово
        # Неизвестные слова не добавляем в выходную последовательность
        if index != 1:
            seq.append(index)
    return seq

Преобразуем все тексты в последовательность кодов слов

In [19]:
banks['Sequences'] = banks.apply(lambda row: text_to_sequence(row['Preprocessed_texts'], word_to_index), axis=1)

In [20]:
banks

,text,label,Preprocessed_texts,Sequences
id,,,,
0,В Альфа-Банке работает замечательная девушка -...,1,"[альфа-банк, работать, замечательный, девушка,...","[61, 44, 896, 76, 194, 1842, 344, 2685, 396, 1..."
1,Оформляя рассрочку в м. Видео в меге тёплый ст...,0,"[оформлять, рассрочка, м., видео, мег, тёплый,...","[254, 836, 1155, 3189, 3865, 2956, 7316, 163, ..."
2,Очень порадовала оперативность работы в банке....,1,"[очень, порадовать, оперативность, работа, бан...","[20, 1033, 890, 30, 2, 461, 175, 3, 613, 1743,..."
3,Имела неосторожность оформить потреб. кредит в...,0,"[иметь, неосторожность, оформить, потреба, кре...","[113, 4790, 70, 2343, 9, 61, 20, 2639, 1264, 3..."
4,Небольшая предыстория: Нашел на сайте MDM банк...,0,"[небольшой, предыстория, найти, сайт, mdm, бан...","[405, 3824, 262, 85, 2, 633, 3, 4866, 2, 283, ..."
...,...,...,...,...
13994,"О высокой надёжности МКБ, порядочности и добро...",1,"[высокий, надёжность, мкб, порядочность, добро...","[388, 2296, 792, 5000, 8128, 7, 3073, 120, 427..."
13995,"Обслуживаюсь в офисе на Чернореченской 42а, ка...",1,"[обслуживаться, офис, чернореченский, 42а, физ...","[349, 26, 9465, 2126, 191, 16, 26, 134, 64, 11..."
13996,Попала сегодня в очень неприятную ситуацию. Ре...,1,"[попасть, сегодня, очень, неприятный, ситуация...","[502, 104, 20, 971, 53, 39, 213, 221, 3, 47, 3..."


## Готовим данные для обучения

Делим данные на наборы для обучения и тестирования

In [21]:
train, test = train_test_split(banks, test_size=0.2)

In [22]:
train

,text,label,Preprocessed_texts,Sequences
id,,,,
3,Имела неосторожность оформить потреб. кредит в...,0,"[иметь, неосторожность, оформить, потреба, кре...","[113, 4790, 70, 2343, 9, 61, 20, 2639, 1264, 3..."
11037,Добрый день! Год назад оформил кредит в банке ...,0,"[добрый, день, год, назад, оформить, кредит, б...","[173, 6, 16, 200, 70, 9, 2, 189, 942, 3253, 94..."
9687,У меня два кредита в Русфинанс Банке. Раньше к...,1,"[кредит, русфинанс, банк, ранний, как-то, особ...","[9, 3077, 2, 387, 526, 1046, 959, 254, 251, 14..."
482,В МВидео я оформила на себя потребительский кр...,0,"[мвидео, оформить, потребительский, кредит, се...","[3805, 70, 432, 9, 538, 2, 107, 408, 136, 78, ..."
5277,Отличный банк!Клиенты банка с 2002 года. В то ...,1,"[отличный, банк, клиент, банк, 2002, год, врем...","[589, 2, 13, 2, 6129, 16, 22, 1551, 439, 237, ..."
...,...,...,...,...
11594,В прошлом году жена делала вклад в отделении н...,0,"[прошлый, год, жена, делать, вклад, отделение,...","[724, 16, 524, 132, 48, 12, 2969, 2201, 1507, ..."
4585,Здравствуйте!4 года назад оформил карту в Моск...,0,"[здравствуйте, 4, год, назад, оформить, карта,...","[510, 247, 16, 200, 70, 3, 1024, 2769, 273, 24..."
1256,Добрый день! По рекомендации горячей линии 06....,0,"[добрый, день, рекомендация, горячий, линия, 0...","[173, 6, 1568, 156, 125, 120, 72, 1185, 803, 5..."


## Определяет и создаем наборы данных

In [23]:
class TextDatasetFromPandas(Dataset):
    def __init__(self, dataframe, feature_col, target_col):
        self.features = dataframe[feature_col].values
        self.labels = dataframe[target_col].values
        
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        tockens = torch.tensor(self.features[idx], dtype=torch.long)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return tockens, label 

In [24]:
train_dataset = TextDatasetFromPandas(dataframe=train,
                                      feature_col='Sequences',
                                      target_col='label')

In [25]:
test_dataset = TextDatasetFromPandas(dataframe=test,
                                      feature_col='Sequences',
                                      target_col='label')

In [26]:
# Функция для паддинга
def collate_fn(batch):
    texts, labels = zip(*batch)
    texts_padded = torch.nn.utils.rnn.pad_sequence(texts, batch_first=True, padding_value=0)
    labels = torch.stack(labels)
    return texts_padded, labels

In [28]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

## Определяем и создаем модель

In [ ]:
class TextClassificationModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super(TextClassificationModel, self).__init__()
        # Слой для создания плотных векторных представлений слов
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, sparse=False)
        # Полносвязный слой. Количество нейронов равно количеству классов
        self.fc = nn.Linear(embed_dim, num_class)


    def forward(self, text):
        embedded = self.embedding(text)
        return self.fc(embedded)

Создаем модель

In [30]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

In [31]:
model = TextClassificationModel(max_words, emsize, num_class).to(device)

## Определяем функцию для обучения




In [32]:
def train_model(model, dataloader, optimizer, criterion):
    epoch_loss = 0
    model.train()

    for batch in dataloader:
        texts, labels = batch
        texts, labels = texts.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        predictions = model(texts)
        loss = criterion(predictions, labels)
          
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()

    
    return epoch_loss / len(dataloader)


def evaluate_model(model, dataloader, criterion):
    epoch_loss = 0
    
    model.eval()
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for batch in dataloader:
            texts, labels = batch
            texts, labels = texts.to(device), labels.to(device)
            
            predictions = model(texts)
            loss = criterion(predictions, labels)
    
            epoch_loss += loss.item()
            
            all_predictions.extend(torch.round(predictions).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return epoch_loss / len(dataloader), all_predictions, all_labels

## Запускаем обучение

In [33]:
# Гиперпараметры
EPOCHS = 10  # Количество эпох

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [34]:
for epoch in range(EPOCHS):
    start_time = time.time()
    
    train_loss = train_model(model, train_loader, optimizer, criterion)
    valid_loss, _, _ = evaluate_model(model, test_loader, criterion)
    
    end_time = time.time()
    epoch_mins, epoch_secs = divmod(end_time - start_time, 60)

## Применяем модель для распознавания отзывов

Вымышленный позитивный отзыв

In [35]:
positive_text = """Брал кредит в Мегабанке на автомобиль. Выдали за один день. Никаких скрытых комиссий и переплат.
У банка удобное мобильное приложение, через которое можно быстро отправить ежемесячный платеж.
Досрочное гасить начал через три месяца. Я доволен оперативностью и удобством. Огромное спасибо!
"""

Вымышленный негативный отзыв

In [36]:
negative_text = """Взял кредит в ТакСебеБанке на автомобиль. В договор включили обязательный контракт
на помощь на дороге, который мне не нужен. Узнал об этом только во время подписания договора, иначе бы отказался.
Альтернативы была страхование жизни, но мне это даже не предложили. Скорее всего, менеджер продвигает
продажи услуг этой компании в ущерб интересов клиента. Как минимум, непорядочно и непрофессионально.
У банка ужасное мобильное приложение, из-за которого с меня взяли штраф 10 тыс.руб. По требованиям
банка после покупки автомобиля в приложении нужно загрузить ПТС. Я загрузил и проверил, что ПТС в приложении есть.
Но через некоторое время ПТС из приложения пропал и с меня взяли штраф. Никому не рекомендую связываться с ТакСебеБанком.
"""

Предварительная обработка текстов отзывов

In [37]:
positive_preprocessed_text = preprocess(positive_text, stop_words, punctuation_marks, morph)

In [38]:
negative_preprocessed_text = preprocess(negative_text, stop_words, punctuation_marks, morph)

In [39]:
positive_preprocessed_text

['брать',
 'кредит',
 'мегабанк',
 'автомобиль',
 'выдать',
 'день',
 'никакой',
 'скрытый',
 'комиссия',
 'переплата',
 'банк',
 'удобный',
 'мобильный',
 'приложение',
 'который',
 'быстро',
 'отправить',
 'ежемесячный',
 'платёж',
 'досрочный',
 'гасить',
 'начать',
 'месяц',
 'довольный',
 'оперативность',
 'удобство',
 'огромный',
 'спасибо']

In [40]:
positive_preprocessed_seq = text_to_sequence(positive_preprocessed_text, word_to_index)

In [41]:
negative_preprocessed_seq = text_to_sequence(negative_preprocessed_text, word_to_index)

In [42]:
positive_preprocessed_seq

[162,
 9,
 942,
 129,
 6,
 79,
 1866,
 78,
 1007,
 2,
 274,
 292,
 747,
 8,
 122,
 195,
 475,
 31,
 345,
 1399,
 266,
 35,
 286,
 890,
 1293,
 299,
 73]

In [43]:
def predict(text):
    with torch.no_grad():
        text = torch.tensor(text).unsqueeze(0).to(device)
        print(text)
        output = model(text)
        return output.argmax(1).item()

In [44]:
predict(positive_preprocessed_seq)

tensor([[ 162,    9,  942,  129,    6,   79, 1866,   78, 1007,    2,  274,  292,
          747,    8,  122,  195,  475,   31,  345, 1399,  266,   35,  286,  890,
         1293,  299,   73]], device='cuda:0')


1

In [45]:
predict(negative_preprocessed_seq)

tensor([[ 121,    9,  942,   29, 1262, 1049, 3518,  300, 1146,    8,  143,  172,
            4,   22,  841,   29,  984,  261, 2837,  233,  407,    4,  124,  920,
           21,   82, 7951, 1069,   45,  328, 1918, 1449,   13,  838, 3150,    2,
         1412,  292,  747,  248,    8,  121,  322,  118, 3603,  438,    2,  189,
          942,  747,   93, 4058,  926, 4058,  398,  926,  747,  319,   22,  926,
          747, 1594,  121,  322,  154,  466, 1114]], device='cuda:0')


0